# Clustering the books: K-Means vs. HDBSCAN

This notebook groups the books into clusters of similar books, using the vectors made by the pipeline
(`python scripts/pipeline.py`). It tries two clustering methods, tunes their settings, picks the best
version of each, and describes what each cluster is about.

**Input:** `data/processed/clustering_inputs/` (written by `scripts/reduce_umap.py`)
**Output:** `data/processed/clustering_results/`: cluster assignments, cluster keywords, scores, figures

**How clusters are scored**

| Score | Question it answers | Good values |
|---|---|---|
| Silhouette | Are clusters tight and well separated? | -1 to 1, higher is better |
| Genre purity | Does each cluster mostly hold one sub-genre? | Compared with a *random* baseline, because purity always rises with more clusters |
| c-TF-IDF purity | Does each book's own text match its cluster's keywords? | 0 to 1, higher is better |
| Noise | HDBSCAN only: share of books it leaves out of every cluster | lower is better |

## 1. Setup

In [ ]:
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import sparse
from sklearn.cluster import HDBSCAN, KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.metrics.cluster import contingency_matrix
from sklearn.preprocessing import normalize

pl.Config.set_tbl_rows(60)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(200)

# Paths (works whether the notebook is opened from notebooks/ or the project folder)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed" / "clustering_inputs"
OUTPUT_DIR = ROOT / "data" / "processed" / "clustering_results"

In [ ]:
# ---- Settings ----
SEED = 42

# Which K-Means and HDBSCAN settings to try
KMEANS_K = list(range(5, 41))  # number of clusters
HDBSCAN_MIN_CLUSTER_SIZE = [10, 15, 20, 30, 40, 50, 75, 100, 150, 200, 300, 500]
HDBSCAN_MIN_SAMPLES = [1, 5, 10, None]  # None = same as min_cluster_size
HDBSCAN_METHODS = ["eom", "leaf"]

# Rules for a usable result
MIN_CLUSTERS, MAX_CLUSTERS = 5, 60  # a useful number of groups for a recommender
MAX_CLUSTER_SHARE = 0.50            # no single cluster may hold more than half the books
MAX_NOISE = 0.30                    # HDBSCAN may leave at most 30% of books unclustered...
NOISE_RELAX_MARGIN = 0.10           # ...unless no setting manages that (then the cap is relaxed, with a warning)

# Scoring details
SILHOUETTE_SAMPLE = 5_000           # silhouette is slow, so it's computed on a sample while tuning
FINAL_SILHOUETTE_SAMPLE = 20_000    # ...and on a bigger sample for the final comparison
PURITY_IGNORE = {"general"}         # books without a specific sub-genre don't count for purity
TOP_N_WORDS = 10                    # keywords shown per cluster
TSNE_MAX_POINTS = 15_000            # the cluster map shows a random sample of this many books

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

## 2. Load the data

In [ ]:
X = np.load(DATA_DIR / "embeddings_umap.npy")  # one row of numbers per book
books = pl.read_parquet(DATA_DIR / "books.parquet")
assert X.shape[0] == books.height, "the embeddings and the book table must have the same number of rows"

N = X.shape[0]
genre = books["genre_label"].to_numpy()
titles = books["title"].to_numpy()
docs = books["document"].fill_null("").to_list()  # description + reviews, for keywords
counts_for_purity = ~np.isin(genre, list(PURITY_IGNORE))

print(f"{N:,} books, {X.shape[1]} numbers each")
books.group_by("genre_label").len().sort("len", descending=True)

## 3. Scoring tools

**c-TF-IDF** finds the words that make each cluster different from the others: it joins all the text in a
cluster into one document, then scores each word by how common it is in that cluster compared with all
clusters. The word counts for every book are computed once here, so scoring each clustering is fast.

In [ ]:
# Words that appear in almost every book review, so they say nothing about a cluster.
EXTRA_STOP_WORDS = {
    "book", "books", "read", "reading", "reader", "readers", "story", "stories", "novel", "novels",
    "author", "authors", "writing", "written", "character", "characters", "series", "page", "pages",
    "chapter", "chapters", "plot", "really", "just", "like", "good", "great", "loved", "love",
    "enjoyed", "liked", "stars", "star", "review", "think", "thought", "lot", "way", "bit", "did",
    "does", "didn", "doesn", "don", "isn", "wasn", "ve", "ll", "time", "end", "ending", "new",
    "make", "makes", "made", "know", "going", "got", "get", "little", "want", "say", "said",
    "mystery", "mysteries", "thriller", "thrillers", "crime", "suspense",
}
vectorizer = CountVectorizer(
    stop_words=sorted(ENGLISH_STOP_WORDS | EXTRA_STOP_WORDS),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]{2,}\b",  # words of 3+ letters
    min_df=5, max_df=0.5, max_features=50_000, dtype=np.float32,
)
WORD_COUNTS = vectorizer.fit_transform(docs).tocsr()  # books x words
VOCAB = vectorizer.get_feature_names_out()
WORD_SHARES = normalize(WORD_COUNTS, norm="l1")       # each book's word counts as shares
print(f"{WORD_COUNTS.shape[1]:,} words in the vocabulary")


def c_tf_idf(labels):
    # Returns the cluster ids and a (clusters x words) matrix of c-TF-IDF scores. Noise (-1) is skipped.
    in_cluster = labels >= 0
    clusters = np.unique(labels[in_cluster])
    membership = sparse.csr_matrix(
        (np.ones(in_cluster.sum(), dtype=np.float32), (np.searchsorted(clusters, labels[in_cluster]), np.flatnonzero(in_cluster))),
        shape=(len(clusters), len(labels)),
    )
    counts = membership @ WORD_COUNTS                       # word counts per cluster
    tf = normalize(counts, norm="l1")                       # how common each word is in the cluster
    word_total = np.asarray(counts.sum(axis=0)).ravel()
    idf = np.log1p((counts.sum() / len(clusters)) / np.maximum(word_total, 1)).astype(np.float32)
    return clusters, tf.multiply(idf).tocsr(), idf


def top_words(scores, n=TOP_N_WORDS):
    return np.argsort(-scores.toarray(), axis=1)[:, :n]


def score(labels, silhouette_sample=SILHOUETTE_SAMPLE):
    # All the scores for one clustering. labels = cluster number per book, -1 = noise.
    in_cluster = labels >= 0
    k = len(np.unique(labels[in_cluster]))
    result = {"n_clusters": k, "noise_frac": float(1 - in_cluster.mean()),
              "max_cluster_share": float(np.bincount(labels[in_cluster]).max() / len(labels)) if k else 1.0}
    if k < 2:
        return result | dict.fromkeys(["silhouette", "genre_purity", "random_purity", "purity_lift", "nmi",
                                       "ctfidf_purity", "topic_diversity"], np.nan)
    # Silhouette (on a sample when there are many books)
    sample = None if in_cluster.sum() <= silhouette_sample else silhouette_sample
    result["silhouette"] = float(silhouette_score(X[in_cluster], labels[in_cluster], sample_size=sample, random_state=SEED))
    # Genre purity: in each cluster, the share of its most common sub-genre (weighted by cluster size)
    use = in_cluster & counts_for_purity
    result["genre_purity"] = float(contingency_matrix(genre[use], labels[use]).max(axis=0).sum() / use.sum())
    # The same, for a random assignment with the same cluster sizes (what "no signal" looks like)
    shuffled = labels.copy()
    shuffled[in_cluster] = np.random.default_rng(SEED).permutation(labels[in_cluster])
    result["random_purity"] = float(contingency_matrix(genre[use], shuffled[use]).max(axis=0).sum() / use.sum())
    result["purity_lift"] = result["genre_purity"] - result["random_purity"]
    result["nmi"] = float(normalized_mutual_info_score(genre[use], labels[use]))
    # c-TF-IDF purity: match every book's text to the closest cluster keyword profile;
    # the score is the share of books matched to their own cluster.
    clusters, cluster_scores, idf = c_tf_idf(labels)
    book_profiles = normalize(WORD_SHARES[in_cluster].multiply(idf).tocsr())
    closest = clusters[(book_profiles @ normalize(cluster_scores).T).toarray().argmax(axis=1)]
    result["ctfidf_purity"] = float((closest == labels[in_cluster]).mean())
    # Topic diversity: share of distinct words among all clusters' top words
    top = top_words(cluster_scores)
    result["topic_diversity"] = float(len(np.unique(top)) / top.size)
    return result

**How the best setting is chosen** (the same rule for both methods):
1. Keep only usable results: between `MIN_CLUSTERS` and `MAX_CLUSTERS` clusters, no cluster above `MAX_CLUSTER_SHARE`
   of the books, and noise ≤ `MAX_NOISE`. If no result meets the noise cap, it's relaxed to the lowest noise any
   usable result reaches plus `NOISE_RELAX_MARGIN`, and a warning is printed.
2. Rank what's left on three scores (silhouette, purity lift, c-TF-IDF purity) and pick the best average rank.
   Ranks are used because the scores are on different scales.

In [ ]:
def pick_best(results, name):
    usable = results.filter(
        pl.col("n_clusters").is_between(MIN_CLUSTERS, MAX_CLUSTERS)
        & (pl.col("max_cluster_share") <= MAX_CLUSTER_SHARE)
        & pl.col("silhouette").is_not_nan()
    )
    if usable.is_empty():
        raise ValueError(f"{name}: no usable setting. Try a wider range of settings.")
    noise_cap = MAX_NOISE
    if usable.filter(pl.col("noise_frac") <= noise_cap).is_empty():
        noise_cap = usable["noise_frac"].min() + NOISE_RELAX_MARGIN
        print(f"WARNING ({name}): no usable setting has noise <= {MAX_NOISE:.0%}. "
              f"The least noisy one leaves {usable['noise_frac'].min():.0%} of books unclustered. "
              f"Relaxing the noise cap to {noise_cap:.0%}.")
    usable = usable.filter(pl.col("noise_frac") <= noise_cap)
    print(f"{name}: {usable.height} of {results.height} settings are usable")
    rank = pl.col("silhouette").rank() + pl.col("purity_lift").rank() + pl.col("ctfidf_purity").rank()
    return usable.with_columns((rank / (3 * pl.len())).round(3).alias("rank_score")).sort("rank_score", descending=True)


SHOW = ["n_clusters", "noise_frac", "max_cluster_share", "silhouette", "genre_purity", "random_purity",
        "purity_lift", "nmi", "ctfidf_purity", "topic_diversity"]

## 4. K-Means: try different numbers of clusters

In [ ]:
start = time.time()
rows = []
for k in KMEANS_K:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    rows.append({"k": k, "inertia": float(km.inertia_)} | score(km.labels_))
kmeans_results = pl.DataFrame(rows)
print(f"Tried {len(KMEANS_K)} values of k in {time.time() - start:.0f}s")
kmeans_ranked = pick_best(kmeans_results, "K-Means")
best_k = int(kmeans_ranked["k"][0])
print(f"Best k: {best_k}")
kmeans_ranked.select(["k", "rank_score", *SHOW[2:]]).head(5).with_columns(pl.selectors.float().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
r = kmeans_results.sort("k")
axes[0].plot(r["k"], r["silhouette"]); axes[0].set_title("Silhouette")
axes[1].plot(r["k"], r["genre_purity"], label="clusters")
axes[1].plot(r["k"], r["random_purity"], "--", color="grey", label="random baseline")
axes[1].set_title("Genre purity"); axes[1].legend()
axes[2].plot(r["k"], r["ctfidf_purity"]); axes[2].set_title("c-TF-IDF purity")
for ax in axes:
    ax.axvline(best_k, color="black", linestyle=":", linewidth=1)
    ax.set_xlabel("number of clusters (k)")
plt.suptitle(f"K-Means scores (dotted line = chosen k = {best_k})", x=0.01, ha="left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "kmeans_scores.png", bbox_inches="tight")
plt.show()

## 5. HDBSCAN: try different settings

In [ ]:
start = time.time()
rows = []
for mcs in HDBSCAN_MIN_CLUSTER_SIZE:
    for ms in HDBSCAN_MIN_SAMPLES:
        if ms is not None and ms > mcs:
            continue
        for method in HDBSCAN_METHODS:
            labels = HDBSCAN(min_cluster_size=mcs, min_samples=ms, cluster_selection_method=method,
                             copy=True, n_jobs=-1).fit_predict(X)
            rows.append({"min_cluster_size": mcs, "min_samples": ms if ms is not None else mcs,
                         "method": method} | score(labels))
hdbscan_results = pl.DataFrame(rows)
print(f"Tried {hdbscan_results.height} settings in {time.time() - start:.0f}s")
hdbscan_ranked = pick_best(hdbscan_results, "HDBSCAN")
best = hdbscan_ranked.row(0, named=True)
HDBSCAN_PARAMS = {"min_cluster_size": best["min_cluster_size"], "min_samples": best["min_samples"],
                  "cluster_selection_method": best["method"]}
print("Best HDBSCAN settings:", HDBSCAN_PARAMS)
hdbscan_ranked.select(["min_cluster_size", "min_samples", "method", "rank_score", *SHOW]).head(5).with_columns(pl.selectors.float().round(3))

## 6. Compare the two best models

HDBSCAN's scores above ignore its noise books, which flatters it: K-Means has to place every book. So HDBSCAN
is scored twice: on its clustered books only, and with each noise book assigned to the nearest cluster.

**Stability:** each model is refit on 5 random 90% samples of the books. The Adjusted Rand Index (ARI)
measures how much the clusters stay the same (1 = identical, 0 = no better than chance).

In [ ]:
kmeans_model = KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit(X)
km_labels = kmeans_model.labels_
hdb_labels = HDBSCAN(**HDBSCAN_PARAMS, copy=True, n_jobs=-1).fit_predict(X)

# HDBSCAN with every noise book moved to the nearest cluster center
hdb_all_labels = hdb_labels.copy()
cluster_ids = np.unique(hdb_labels[hdb_labels >= 0])
centers = np.vstack([X[hdb_labels == c].mean(axis=0) for c in cluster_ids])
noise = hdb_labels < 0
if noise.any():
    distances = ((X[noise, None, :] - centers[None, :, :]) ** 2).sum(axis=-1)
    hdb_all_labels[noise] = cluster_ids[distances.argmin(axis=1)]

comparison = pl.DataFrame([
    {"model": f"K-Means (k={best_k})"} | score(km_labels, FINAL_SILHOUETTE_SAMPLE),
    {"model": "HDBSCAN (clustered only)"} | score(hdb_labels, FINAL_SILHOUETTE_SAMPLE),
    {"model": "HDBSCAN (all books)"} | score(hdb_all_labels, FINAL_SILHOUETTE_SAMPLE),
])


def stability(fit, reference):
    rng = np.random.default_rng(SEED)
    scores = []
    for _ in range(5):
        idx = np.sort(rng.choice(N, size=int(0.9 * N), replace=False))
        scores.append(adjusted_rand_score(reference[idx], fit(X[idx])))
    return float(np.mean(scores)), float(np.std(scores))


km_stab = stability(lambda Z: KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit_predict(Z), km_labels)
hdb_stab = stability(lambda Z: HDBSCAN(**HDBSCAN_PARAMS, copy=True, n_jobs=-1).fit_predict(Z), hdb_labels)
print(f"Stability (ARI): K-Means {km_stab[0]:.3f} ± {km_stab[1]:.3f} | HDBSCAN {hdb_stab[0]:.3f} ± {hdb_stab[1]:.3f}")
comparison.with_columns(pl.selectors.float().round(3))

## 7. What is each cluster about?

In [ ]:
def describe_clusters(labels, model_name):
    clusters, cluster_scores, _ = c_tf_idf(labels)
    top = top_words(cluster_scores)
    rows = []
    for i, c in enumerate(clusters):
        members = labels == c
        member_genres = genre[members & counts_for_purity]
        if member_genres.size:
            names, n = np.unique(member_genres, return_counts=True)
            main_genre, share = names[n.argmax()], n.max() / member_genres.size
        else:
            main_genre, share = "-", np.nan
        rows.append({"model": model_name, "cluster": int(c), "size": int(members.sum()),
                     "top_words": ", ".join(VOCAB[top[i]]), "top_genre": main_genre,
                     "genre_share": round(float(share), 2), "example_titles": " | ".join(titles[members][:3])})
    return pl.DataFrame(rows).sort("size", descending=True)


km_topics = describe_clusters(km_labels, "kmeans")
hdb_topics = describe_clusters(hdb_labels, "hdbscan")
print(f"K-Means: {km_topics.height} clusters")
km_topics.drop("model")

In [ ]:
print(f"HDBSCAN: {hdb_topics.height} clusters, plus {np.mean(hdb_labels < 0):.0%} of books as noise")
hdb_topics.drop("model")

## 8. Cluster map

In [ ]:
# A 2-D picture of the books (t-SNE), for looking only; the models don't use it.
# Grey = all books, color = the 3 largest clusters. Large datasets are shown as a random sample.
show = np.sort(np.random.default_rng(SEED).choice(N, TSNE_MAX_POINTS, replace=False)) if N > TSNE_MAX_POINTS else np.arange(N)
XY = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(X[show])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, labels, topics, name in [(axes[0], km_labels, km_topics, f"K-Means (k={best_k})"),
                                 (axes[1], hdb_labels, hdb_topics, "HDBSCAN")]:
    labels = labels[show]
    ax.scatter(*XY.T, s=3, color="lightgrey")
    for row, color in zip(topics.head(3).iter_rows(named=True), ["tab:blue", "tab:orange", "tab:green"]):
        m = labels == row["cluster"]
        ax.scatter(*XY[m].T, s=6, color=color, label=f"#{row['cluster']} ({row['size']}): {row['top_words'].split(', ')[0]}, ...")
    ax.set_title(name, loc="left"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figures" / "cluster_map.png", bbox_inches="tight")
plt.show()

## 9. Save the results

In [ ]:
books.select("book_id", "title", "genre_label").with_columns(
    pl.Series("kmeans_cluster", km_labels.astype(np.int32)),
    pl.Series("hdbscan_cluster", hdb_labels.astype(np.int32)),          # -1 = noise
    pl.Series("hdbscan_cluster_all", hdb_all_labels.astype(np.int32)),  # noise moved to the nearest cluster
).write_parquet(OUTPUT_DIR / "cluster_assignments.parquet")
pl.concat([km_topics, hdb_topics]).write_parquet(OUTPUT_DIR / "cluster_topics.parquet")
kmeans_results.write_parquet(OUTPUT_DIR / "tuning_kmeans.parquet")
hdbscan_results.write_parquet(OUTPUT_DIR / "tuning_hdbscan.parquet")
joblib.dump(kmeans_model, OUTPUT_DIR / "kmeans_model.joblib")  # can place new books with .predict()

summary = {
    "inputs": {"n_books": N, "dims": X.shape[1]},
    "kmeans": {"params": {"n_clusters": best_k}, "stability_ari": km_stab[0]},
    "hdbscan": {"params": HDBSCAN_PARAMS, "stability_ari": hdb_stab[0]},
    "comparison": comparison.to_dicts(),
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("Saved to", OUTPUT_DIR)